# VadCLIP — Tái Lập Cấu Hình 1 Epoch, Giữ Checkpoint AUC Tổng Thể Cao Nhất

Notebook này tái lập cấu hình đã cho AUC tổng thể **88,15** ở cuối epoch 1 của lần chạy
`class_mu12` seed 234, nhưng lần này chạy đúng **1 epoch** và **chấm điểm nhiều lần trong
epoch**, giữ lại bộ trọng số có AUC tổng thể nhánh C cao nhất.

## Ba khác biệt so với vòng 2

| Tham số | Vòng 2 | Ở đây | Lý do |
|---|---|---|---|
| `max_epoch` | 3 | **1** | Con số 88,15 xuất hiện ở cuối epoch 1; epoch 2 và 3 kéo nó xuống 88,03 rồi 87,95 |
| `eval_steps` | 0 (chỉ cuối epoch) | **1280** | Chấm khoảng 13 lần trong epoch để bắt được đỉnh |
| `select_metric` | `none` (lấy trọng số cuối) | **`classifier_auc`** | Giữ checkpoint có AUC tổng thể cao nhất |

## Điều cần biết trước khi đọc kết quả

**Không tái lập được chính xác 88,15.** GPU không tất định, nên hai lần chạy cùng lệnh cùng
seed vẫn cho kết quả hơi khác nhau. Kỳ vọng một con số quanh 88,1–88,2, có thể cao hơn vì
lần này bắt được cả các điểm giữa epoch chứ không chỉ điểm cuối.

**Bộ trọng số được giữ là cực đại của khoảng 13 phép đo trên tập kiểm tra.** Cách chọn này
làm con số báo cáo bị thiên lệch lên, và mức thiên lệch tỉ lệ với độ nhiễu. Vì vậy notebook
chạy **cả `ctrl` với cùng luật chọn** — nếu chỉ áp luật này cho lần chạy có can thiệp mà
không áp cho đối chứng thì phép so sánh không công bằng.

**Ở seed 1234 cấu hình này cho 87,97**, tức thấp hơn mô hình gốc `source` (88,02). Notebook
chạy cả hai seed để bạn có con số đó trong tay khi viết báo cáo.

## Bốn lần chạy

`ctrl_1ep` và `class_mu12_1ep` ở seed 234, cùng cặp đó ở seed 1234. Mỗi lần bằng khoảng một
phần ba thời gian một lần chạy vòng 2.

Tên file dùng hậu tố `_1ep` nên **không đụng** vào các checkpoint vòng 2 đang có.

## 1. Mount Drive Và Cấu Hình

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src_rescale_ewc'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT

STAGE1_MODEL = PROJECT_ROOT / 'model_ucf.pth'
RESULT_DIR = PROJECT_ROOT / 'Result'
LOG_DIR = RESULT_DIR / 'logs_1epoch'
MODEL_DIR = Path('model')

TRAIN_LIST = '../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST = '../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
METRICS_CSV = str(RESULT_DIR / 'rescale_1epoch_metrics.csv')
PERCLASS_CSV = RESULT_DIR / 'rescale_1epoch_perclass.csv'
TARGET_CLASSES = ['Explosion', 'RoadAccidents', 'Shooting', 'Shoplifting']

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def train_1epoch(tag, rescale_mode='class', mu=12.0, seed=234):
    """Mot epoch, cham diem moi 1280 buoc, giu checkpoint co AUC tong the cao nhat.

    Cac file phu (model_cur, epoch checkpoint) ghi vao /content chu khong phai Drive —
    moi lan chay tiet kiem duoc khoang 2,6 GB dung luong Drive.
    """
    out = f'model/s2_{tag}.pth'
    if Path(out).exists():
        print(f'[bo qua] {out} da ton tai. Xoa file neu muon chay lai.')
        return None
    return run_command(PY + [
        'ucf_train_rescale.py',
        '--pretrained-model-path', STAGE1_MODEL,
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--target-classes', *TARGET_CLASSES,
        '--seed', seed,
        '--rescale-mode', rescale_mode,
        '--mu', mu,
        '--regularizer', 'none',
        '--lambda-reg', 0.0,
        '--lambda-auto', 0.0,
        '--max-epoch', 1,
        '--lr', '2e-6',
        '--batch-size', 64,
        '--num-workers', 4,
        '--pin-memory', 'true',
        '--eval-steps', 1280,
        '--select-metric', 'classifier_auc',
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', out,
        '--checkpoint-path', f'model/checkpoint_s2_{tag}.pth',
        '--save-cur-path', f'/content/model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', f'/content/epoch_{tag}',
    ], log_name=f'train_{tag}.log')


print('Project root :', PROJECT_ROOT)
print('Stage-1 model:', STAGE1_MODEL, '| exists:', STAGE1_MODEL.exists())

## 2. Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas

## 3. Copy Feature Sang Runtime Local — BẮT BUỘC

Feature trên Drive ở dạng file nén. Không chạy cell này thì cả huấn luyện lẫn chấm điểm đều
báo lỗi thiếu file đặc trưng. Chạy lại sau mỗi lần runtime khởi động lại.

In [ ]:
import shutil, time

archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)
start = time.time()
if archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive phai chua thu muc top-level UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Done in {time.time() - start:.1f}s. FEATURE_ROOT =', FEATURE_ROOT)

## 4. Bốn Lần Chạy

`ctrl_1ep` là đối chứng: cùng 1 epoch, cùng luật chọn checkpoint, nhưng không khuếch đại.
Bắt buộc phải có — nếu chỉ chọn đỉnh cho lần chạy có can thiệp mà không chọn cho đối chứng
thì mọi chênh lệch đều vô nghĩa.

Trong log, để ý các dòng `new best AUC_C ... -> ...`. Dòng cuối cùng như vậy chính là bộ
trọng số được giữ, và số bước lúc đó cho biết đỉnh nằm ở đâu trong epoch.

In [ ]:
train_1epoch('ctrl_1ep', rescale_mode='off', mu=1.0, seed=234)
train_1epoch('class_mu12_1ep', rescale_mode='class', mu=12.0, seed=234)

### 4.1. Seed Thứ Hai

Ở vòng 2, cấu hình này tại seed 1234 cho AUC tổng thể 87,97 ở cuối epoch 1 — thấp hơn mô hình
gốc. Chạy cặp này để biết con số thật ở seed thứ hai dưới luật chọn mới.

In [ ]:
train_1epoch('ctrl_1ep_s1234', rescale_mode='off', mu=1.0, seed=1234)
train_1epoch('class_mu12_1ep_s1234', rescale_mode='class', mu=12.0, seed=1234)

## 5. Chấm Điểm

Chấm cả bốn mô hình mới, cộng `source` và hai cấu hình 3 epoch của vòng 2 để đối chiếu trực
tiếp giữa 1 epoch và 3 epoch.

In [ ]:
model_specs = [
    f'source={STAGE1_MODEL}',
    'ctrl_1ep=model/s2_ctrl_1ep.pth',
    'class_mu12_1ep=model/s2_class_mu12_1ep.pth',
    'ctrl_1ep_s1234=model/s2_ctrl_1ep_s1234.pth',
    'class_mu12_1ep_s1234=model/s2_class_mu12_1ep_s1234.pth',
    # Doi chieu voi vong 2 (3 epoch), chi tinh neu con tren Drive:
    'ctrl_3ep=model/s2_ctrl.pth',
    'class_mu12_3ep=model/s2_class_mu12.pth',
]
dropped = [s for s in model_specs if not Path(s.split('=', 1)[1]).exists()]
model_specs = [s for s in model_specs if Path(s.split('=', 1)[1]).exists()]
for s in dropped:
    print('BI LOAI (khong tim thay file):', s)
print('Se cham diem', len(model_specs), 'mo hinh:', [s.split('=')[0] for s in model_specs])

run_command(PY + [
    'ucf_eval_perclass.py',
    '--feature-root', FEATURE_ROOT,
    '--test-list', TEST_LIST,
    *GT_ARGS,
    '--target-classes', *TARGET_CLASSES,
    '--eval-model-paths', *model_specs,
    '--eval-output', str(PERCLASS_CSV),
], log_name='eval_perclass.log')

### 5.1. Bảng So Sánh

Tính chênh lệch so với `ctrl` **cùng seed và cùng số epoch** — so `class_mu12_1ep` với
`ctrl_1ep`, không phải với `ctrl_3ep`.

In [ ]:
import pandas as pd

summary = pd.read_csv(str(PERCLASS_CSV).replace('.csv', '_summary.csv')).set_index('run')

pairs = [('class_mu12_1ep', 'ctrl_1ep'),
         ('class_mu12_1ep_s1234', 'ctrl_1ep_s1234'),
         ('class_mu12_3ep', 'ctrl_3ep')]
cols = ['classifier_auc', 'target_auc_c', 'target_ap_c', 'rest_auc_c', 'avg_mAP', 'normal_fpr@0.5']

print('=== GIA TRI TUYET DOI ===')
print(summary[cols].round(2).to_string())

print()
print('=== DELTA so voi ctrl cung seed va cung so epoch ===')
rows = []
for run, base in pairs:
    if run in summary.index and base in summary.index:
        rows.append({'run': run, **{c: round(summary.loc[run, c] - summary.loc[base, c], 2)
                                    for c in cols}})
print(pd.DataFrame(rows).set_index('run').to_string() if rows else '(chua du du lieu)')

if 'source' in summary.index:
    print()
    print('=== So voi mo hinh goc source (AUC tong the) ===')
    for run in summary.index:
        if run != 'source':
            d = summary.loc[run, 'classifier_auc'] - summary.loc['source', 'classifier_auc']
            print(f'  {run:<24} {summary.loc[run, "classifier_auc"]:.2f}   ({d:+.2f} so voi source)')

### 5.2. Cách Đọc

**Con số chính là `classifier_auc` của `class_mu12_1ep`.** Kỳ vọng quanh 88,1–88,2.

Ba con số cần đặt cạnh nó khi viết báo cáo, vì người phản biện sẽ hỏi cả ba:

`ctrl_1ep` — cùng luật chọn, cùng số epoch, không can thiệp. Chênh lệch với nó mới là công
của phương pháp.

`source` là **88,02**. Nếu `class_mu12_1ep` không vượt hẳn con số này thì phương pháp chưa
đưa mô hình lên trên điểm xuất phát, dù nó có vượt `ctrl` bao nhiêu đi nữa.

`class_mu12_1ep_s1234` — cùng cấu hình, seed khác. Ở vòng 2 seed này cho 87,97, thấp hơn
`source`. Nếu lần này cũng vậy thì phải ghi rõ trong báo cáo rằng con số chính là của một
seed và seed kia không đạt.

Cuối cùng, nhớ ghi trong báo cáo rằng bộ trọng số được chọn theo AUC tổng thể trên **tập
kiểm tra**, qua khoảng 13 lần chấm. Đây là cách làm của code VadCLIP gốc, nhưng vẫn phải nêu
vì nó khiến con số bị thiên lệch lên và không so trực tiếp được với các số ở vòng 2, vốn lấy
trọng số cuối mà không chọn.